In [1]:
# Conversational RAG Demo with Memory - Google Colab Ready
# This demo shows: RAG + Conversation Memory + Multi-turn Context

# Step 1: Install required packages
print(" Installing packages (this may take 1-2 minutes)...")
print("   (Dependency warnings are normal in Colab and can be ignored)\n")
!pip install -q langchain langchain-community langchain-google-genai sentence-transformers chromadb google-generativeai 2>&1 | grep -v "dependency conflicts\|incompatible\|ERROR: pip's dependency" || true
print("\n Installation complete!\n")

 Installing packages (this may take 1-2 minutes)...
   (Dependency warnings are normal in Colab and can be ignored)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━

In [2]:
# Step 2: Set up API Key
import getpass
import os
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY") or userdata.get("GEM_API_KEY")
if not api_key:
    raise RuntimeError("Add your Gemini key to Colab userdata as GEMINI_API_KEY (or GEM_API_KEY).")
os.environ["GOOGLE_API_KEY"] = api_key
print("Gemini API key configured.")

Gemini API key configured.


In [3]:
# Step 3: Import libraries
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate
from langchain.schema import Document

print(" Starting Conversational RAG Pipeline with Memory")
print("="*60)

 Starting Conversational RAG Pipeline with Memory


In [4]:
# Step 4: Create sample documents (expanded knowledge base)
print("\n Step 1: Creating knowledge base...")
documents = [
    Document(page_content="LangChain is a framework for developing applications powered by language models. It provides tools for document loading, splitting, embeddings, chains, and memory management.",
             metadata={"source": "doc1", "topic": "langchain"}),
    Document(page_content="RAG stands for Retrieval-Augmented Generation. It combines retrieval of relevant documents with text generation. The system first searches for relevant information, then uses that context to generate informed responses.",
             metadata={"source": "doc2", "topic": "rag"}),
    Document(page_content="Vector databases store embeddings and allow for semantic similarity search. They convert text into numerical vectors that capture meaning. Popular vector databases include ChromaDB, Pinecone, and Weaviate.",
             metadata={"source": "doc3", "topic": "vectordb"}),
    Document(page_content="ConversationBufferMemory in LangChain stores the entire conversation history. This allows the system to maintain context across multiple turns and reference previous messages.",
             metadata={"source": "doc4", "topic": "memory"}),
    Document(page_content="Conversational RAG combines retrieval, generation, and memory. Unlike basic RAG, it can handle follow-up questions, maintain context, and build on previous responses.",
             metadata={"source": "doc5", "topic": "conversational-rag"}),
    Document(page_content="Google Gemini is a powerful multimodal AI model that excels at text generation, reasoning, and following instructions. Gemini Flash 2.0 is the latest and fastest version.",
             metadata={"source": "doc6", "topic": "gemini"}),
    Document(page_content="Memory types in LangChain include ConversationBufferMemory (stores everything), ConversationSummaryMemory (stores summaries), and ConversationBufferWindowMemory (stores last K messages).",
             metadata={"source": "doc7", "topic": "memory"}),
    Document(page_content="ChromaDB is an open-source embedding database that works well with LangChain. It's lightweight, fast, and perfect for development and prototyping.",
             metadata={"source": "doc8", "topic": "vectordb"}),
]


 Step 1: Creating knowledge base...


In [5]:
# Step 5: Split documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
splits = text_splitter.split_documents(documents)
print(f" Created {len(splits)} document chunks")

 Created 8 document chunks


In [6]:
# Step 6: Create embeddings
print("\n Step 2: Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print(" Embedding model loaded")


 Step 2: Loading embedding model...


/tmp/ipython-input-130396009.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Embedding model loaded


In [7]:
# Step 7: Create vector store
print("\n Step 3: Creating vector database...")
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="conversational_rag_demo"
)
print(" Vector database created")


 Step 3: Creating vector database...
 Vector database created


In [8]:
# Step 8: Set up retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [9]:
# Step 9: Initialize Gemini LLM
print("\n Step 4: Initializing Gemini Flash 2.0...")
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-exp",
    temperature=0.7,
    top_p=0.9,
    max_output_tokens=1024,
)
print(" Gemini Flash 2.0 ready")


 Step 4: Initializing Gemini Flash 2.0...
 Gemini Flash 2.0 ready


In [10]:
# Step 10: Create Conversation Memory
print("\n Step 5: Setting up Conversation Memory...")
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)
print(" Memory initialized")


 Step 5: Setting up Conversation Memory...
 Memory initialized


/tmp/ipython-input-3767542796.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [11]:
# Step 11: Create custom prompt for conversational RAG
condense_question_prompt = PromptTemplate(
    template="""Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question.

Chat History:
{chat_history}

Follow Up Question: {question}

Standalone Question:""",
    input_variables=["chat_history", "question"]
)

qa_prompt = PromptTemplate(
    template="""You are a helpful AI assistant. Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Provide clear, detailed, and conversational answers. If the question refers to previous messages, use the chat history to maintain context.

Context: {context}

Question: {question}

Helpful Answer:""",
    input_variables=["context", "question"]
)

In [12]:
# Step 12: Create Conversational RAG Chain
print("\n Step 6: Building Conversational RAG chain...")
conversational_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    condense_question_prompt=condense_question_prompt,
    combine_docs_chain_kwargs={"prompt": qa_prompt},
    return_source_documents=True,
    verbose=False
)
print(" Conversational RAG chain ready!")


 Step 6: Building Conversational RAG chain...
 Conversational RAG chain ready!


In [13]:
# Step 13: Demo conversation function
print("\n" + "="*60)
print(" CONVERSATIONAL RAG DEMONSTRATION")
print("="*60)

def chat(question, show_sources=True):
    """
    Chat with the conversational RAG system
    """
    print(f"\n You: {question}")

    result = conversational_chain({"question": question})

    print(f"\n Assistant: {result['answer']}")

    if show_sources:
        print(f"\n Sources used: {[doc.metadata['source'] for doc in result['source_documents']]}")

    print("\n" + "-"*60)

    return result


 CONVERSATIONAL RAG DEMONSTRATION


In [14]:
# Step 14: Run a demo conversation
print("\n Running Demo Conversation...")
print("\nThis demo shows how the system maintains context across multiple turns:\n")

# Turn 1: Initial question
chat("What is RAG?")

# Turn 2: Follow-up question (uses context from Turn 1)
chat("How does it work exactly?")

# Turn 3: Another follow-up (builds on previous context)
chat("What's the difference between basic RAG and conversational RAG?")

# Turn 4: Reference to earlier conversation
chat("You mentioned memory earlier. Tell me more about the different types.")

# Turn 5: Complex follow-up
chat("Which memory type would you recommend for my use case?")



 Running Demo Conversation...

This demo shows how the system maintains context across multiple turns:


 You: What is RAG?


/tmp/ipython-input-3546371594.py:12: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = conversational_chain({"question": question})



 Assistant: RAG stands for Retrieval-Augmented Generation. It's a cool technique that combines grabbing relevant info with creating text. Basically, the system first searches for helpful documents and then uses that info to generate well-informed responses.

 Sources used: ['doc2', 'doc5', 'doc8']

------------------------------------------------------------

 You: How does it work exactly?

 Assistant: Retrieval-Augmented Generation (RAG) works by combining information retrieval with text generation. First, it retrieves relevant documents or information from a knowledge base based on the user's query. Then, it uses this retrieved context to generate a more informed and accurate response.

 Sources used: ['doc2', 'doc5', 'doc4']

------------------------------------------------------------

 You: What's the difference between basic RAG and conversational RAG?

 Assistant: The key differences between basic RAG and conversational RAG systems lie in their ability to handle context and fo

{'question': 'Which memory type would you recommend for my use case?',
 'chat_history': [HumanMessage(content='What is RAG?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="RAG stands for Retrieval-Augmented Generation. It's a cool technique that combines grabbing relevant info with creating text. Basically, the system first searches for helpful documents and then uses that info to generate well-informed responses.", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='How does it work exactly?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Retrieval-Augmented Generation (RAG) works by combining information retrieval with text generation. First, it retrieves relevant documents or information from a knowledge base based on the user's query. Then, it uses this retrieved context to generate a more informed and accurate response.", additional_kwargs={}, response_metadata={}),
  HumanMessage(content="What's the difference between basic